In [ ]:
import pandas as pd
import numpy as np
import _warnings
# ======================================
# 1. Load Dataset
# ======================================
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# ======================================
# 2. Fix TotalCharges
# ======================================
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# ======================================
# 3. Handle Missing Values
# ======================================
df.dropna(inplace=True)

# ======================================
# 4. Remove Unnecessary Columns
# ======================================
df.drop("customerID", axis=1, inplace=True)

# ======================================
# 5. Feature Engineering
# ======================================

# Tenure Groups
bins = [-1, 12, 24, 36, 48, 60, 72]

labels = [
    "0-12",
    "13-24",
    "25-36",
    "37-48",
    "49-60",
    "61-72"
]

df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=bins,
    labels=labels
)

# Average Monthly Spend
df["AvgMonthlySpend"] = (
    df["TotalCharges"] /
    df["tenure"]
)

df["AvgMonthlySpend"].replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

df["AvgMonthlySpend"].fillna(
    df["AvgMonthlySpend"].median(),
    inplace=True
)

# ======================================
# 6. Binary Encoding
# ======================================

binary_cols = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "Churn"
]

for col in binary_cols:
    df[col] = df[col].map(
        {
            "Yes": 1,
            "No": 0
        }
    )

# ======================================
# 7. One-Hot Encoding
# ======================================

df_final = pd.get_dummies(
    df,
    drop_first=True
)

# ======================================
# 8. Final Validation
# ======================================

df_final.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

df_final.fillna(
    df_final.median(
        numeric_only=True
    ),
    inplace=True
)

print("\nData Quality Check")
print("-"*30)
print(
    "Missing Values:",
    df_final.isna().sum().sum()
)

print(
    "Infinite Values:",
    np.isinf(df_final).sum().sum()
)

# ======================================
# 9. Save Clean Dataset
# ======================================

df_final.to_csv(
    "clean_customer_churn.csv",
    index=False
)

print("\n? Clean CSV saved successfully")
print("Final Shape:", df_final.shape)

In [ ]:
import joblib

feature_names = joblib.load("feature_names.pkl")

print("Total Features:", len(feature_names))

for feature in feature_names:
    print(feature)